## 네이버 VIBE data 분석


In [ ]:
# CSV 자동 준비 (Colab · JupyterLite · 로컬 공통)
# 파일이 없으면 GitHub raw에서 받아 cwd에 저장 → pd.read_csv("파일명") 그대로 동작
import sys
from pathlib import Path

def ensure_csv(filename, repo_path):
    candidates = [
        Path(filename),
        Path(repo_path),
        Path("files") / repo_path,
        Path("/files") / repo_path,
        Path(repo_path).name,
    ]
    for c in candidates:
        try:
            if c.is_file():
                if not Path(filename).exists():
                    Path(filename).write_bytes(c.read_bytes())
                return filename
        except Exception:
            pass
    url = f"https://raw.githubusercontent.com/aaronlee09-max/informatics/main/{repo_path}"
    try:
        if sys.platform == "emscripten":
            from pyodide.http import open_url
            Path(filename).write_text(open_url(url).read(), encoding="utf-8")
        else:
            import urllib.request
            urllib.request.urlretrieve(url, filename)
        print("CSV 준비:", filename)
        return filename
    except Exception as e:
        print("CSV 준비 실패 → URL 직접 사용:", url, e)
        return url

ensure_csv('navermusic_genre.csv', 'informatics/ch2-3/navermusic_genre.csv')

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
# 연령별 데이터 불러오기
df = pd.read_csv('navermusic_age.csv')
df.head()


In [ ]:
df.shape
df.info()


In [ ]:
# 1-가. 가장 많은 재생수 음악
sorted = df.sort_values(by='playnum', ascending=False)
plt.figure(figsize=(12,5))
plt.bar(sorted['title'][:10], sorted['playnum'][:10])
plt.xticks(rotation=45, ha='right')
plt.show()


In [ ]:
# 1-나. 10대가 가장 많이 들은 음악
df['10대 재생수'] = df['playnum'] * df['10']
df.sort_values(by='10대 재생수', ascending=False).head(10)


In [ ]:
# 1-다. 10대가 가장 많이 들은 가수
by_artist = df.groupby('artist')['10대 재생수'].sum().sort_values(ascending=False).head(10)
plt.bar(by_artist.index, by_artist.values)
plt.xticks(rotation=45, ha='right')
plt.show()


In [ ]:
genre_df = pd.read_csv('navermusic_genre.csv')
genre_df


In [ ]:
genres =['ballad','dance','hiphop','pop','rock','carol']
plt.figure(figsize=(12,6))
for genre in genres:
    plt.plot(genre_df['month'], genre_df[genre], marker='o', label=genre)
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.show()
